# Gemma Falsification Review v3 (Chat Template Fix)

RCA: v1/v2 failed because Gemma instruction-tuned models require the
chat template (`<start_of_turn>` markers). Raw text input produces
empty generation. This version uses `apply_chat_template`.


In [ ]:
!pip install -q "transformers>=5.5" 2>&1 | tail -1
import transformers
print(f'transformers: {transformers.__version__}')


In [ ]:
import os
def find_model():
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'config.json' in files and any(f.endswith('.safetensors') for f in files):
            return root
    return None
MODEL_PATH = find_model()
print('model at:', MODEL_PATH)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, device_map='auto')
model.eval()
print('loaded')


In [ ]:
def ask_gemma(question, max_tokens=600):
    messages = [{'role': 'user', 'content': question}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Smoke test
print(ask_gemma('What is 2+2? Answer in one sentence.', max_tokens=50))


## Red-team questions


In [ ]:
q1 = '''A quantum transpiler optimizes circuits before execution on hardware. Two circuits are byte-identical after transpilation except for one rotation angle value (0 vs pi). Can the transpiler have introduced a systematic difference between them? Explain in 4 sentences.'''
print('=== Q1: Transpiler ===')
print(ask_gemma(q1))


In [ ]:
q2 = '''In quantum computing, readout error is modeled as a classical bit-flip channel: measured contrast dM = (1-e0-e1)*dp where dp is the true population difference and e0,e1 are readout error rates. If dp=0, can readout error alone produce nonzero measured contrast? What assumption does this rely on? Answer in 4 sentences.'''
print('=== Q2: Readout linearity ===')
print(ask_gemma(q2))


In [ ]:
q3 = '''A superconducting qubit shows an unmodeled 30 kHz detuning. One hypothesis is AC Stark shift from residual photons in the readout resonator (2*chi*nbar). Another is intrinsic qubit frequency offset. What experiment distinguishes these? Describe the key signature difference in 4 sentences.'''
print('=== Q3: AC Stark vs intrinsic ===')
print(ask_gemma(q3))


In [ ]:
q4 = '''Quantum measurement is sometimes modeled as a classical bit-flip channel, but real measurement involves the qubit entangling with a readout resonator. If the pre-measurement quantum states differ between two experiments, could the measurement process itself introduce a differential effect beyond classical readout error? Answer in 4 sentences.'''
print('=== Q4: Quantum measurement backaction ===')
print(ask_gemma(q4))


## Verdict
Gemma's answers above. Q4 addresses the Claim 4 tension directly:
if measurement backaction differs between bright/dark, the classical
readout linearity argument has a gap.
